# KNSB Thermochemical Justification

## 1. Setting up the environment

I'll be using a powerful tool known as [PyProPEP](https://github.com/jonnydyer/pypropep/tree/master) for this task. It's a Python interface to [CProPEP](https://rocketworkbench.sourceforge.net/) (*an improvement on ForTran ProPEP*). It calculates everything accurately and efficiently. Also ensure you've set up Jupyter on your machine.

First, install the libraries then we're ready to go.

---
## 2. A little bit on it's Stoichiometry...
We'll now solve for the following equation:

$$
KNO_{3\left(s\right)}+C_6H_{14}O_{6\left(s\right)} \rightarrow CO_{2\left(g\right)}+CO_{\left(g\right)}+H_2O_{\left(g\right)}+H_{2\left(g\right)}+N_{2\left(g\right)}+K_2CO_{3\left(g\right)}+KOH_{\left(g\right)}
$$

> This is what annoys me with this equation. It's too spread out and at the high temperature i.e. **Adiabatic Flame Temperature** (AFT) - *that's ranges from $2100\:K$ to $3500\:K$ depending on the composite in question.* - one can intuitively tell that these extra products will collapse into the following:
> $$
\boxed{
C_6H_{14}O_6+6O_2 \rightarrow 6CO_2+7H_2O
}
> $$

A necessary GCD reduction step after solving the null-space problem as ensures you get the minimal positive integer coefficients that represent the equation as the free variable value will be "kinda wonky".

This is implemented in `stoichiometric_justification.ipynb`.

---
## 3. Now the fun part...

I'll first start by importing the various libraries we need for this task. 


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

import operator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import Math, Latex
import pprint as ppr

# Fix deprecated collections imports
import collections.abc
import collections
if not hasattr(collections, 'MutableMapping'):
    collections.MutableMapping = collections.abc.MutableMapping
if not hasattr(collections, 'Mapping'):
    collections.Mapping = collections.abc.Mapping
if not hasattr(collections, 'Sequence'):
    collections.Sequence = collections.abc.Sequence

import pypropep as ppp

We'll initialize `pypropep` and set up our plotting as follows:

In [ ]:
RHO_KNO3     = 2.109
RHO_SORBITOL = 1.489
P_CHAMBER_ATM = 68.0
P_EXIT_ATM    = 1.0

plt.style.use(u'ggplot')
plt.rcParams['figure.figsize'] = (10,6)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})

ppp.init()

Loaded 1921 thermo species
Loaded 1031 propellants


Confirming that we have our oxidiser...

In [ ]:
kno3 = ppp.PROPELLANTS['POTASSIUM NITRATE']
Math(kno3.formula(tex=True))

<IPython.core.display.Math object>

...and our fuel...

In [ ]:
sorb = ppp.PROPELLANTS['SORBITOL']
Math(sorb.formula(tex=True))

<IPython.core.display.Math object>

Then I'll define my global variables that will be used throughout the code. For this we'll need the **mechanism** which points to a yaml file that contains the details of the Grains in question and to define some lists like the **flame temp.** and **specific impulse** that will contain the calculated values from our script

We'll then parse the data in the CSV file and get the masses of $KNO_3$ and $C_6H_{14}O_6$ and store them as a dataframe.

In [ ]:
df = pd.read_csv("../share/csv/Thermochemical_Justification.csv")

ppr.pprint(df.dtypes)
ppr.pprint(df.isna().sum())

df['Mass KNO3 (g)']    = pd.to_numeric(df['Mass KNO3 (g)'], errors='coerce')
df['Mass Sorbitol (g)'] = pd.to_numeric(df['Mass Sorbitol (g)'], errors='coerce')
df = df.dropna(subset=['Mass KNO3 (g)','Mass Sorbitol (g)'])

df.head()

KNO3 (%)             float64
Sorbitol (%)         float64
Mass KNO3 (g)        float64
Mass Sorbitol (g)    float64
C*                   float64
Density              float64
Chamber Cp/Cv        float64
Chamber Temp         float64
Isp*                 float64
dtype: object
KNO3 (%)             0
Sorbitol (%)         0
Mass KNO3 (g)        0
Mass Sorbitol (g)    0
C*                   6
Density              6
Chamber Cp/Cv        6
Chamber Temp         6
Isp*                 6
dtype: int64


,KNO3 (%),Sorbitol (%),Mass KNO3 (g),Mass Sorbitol (g),C*,Density,Chamber Cp/Cv,Chamber Temp,Isp*
0,50.00,50.00,726.80,986.83,NaN,NaN,NaN,NaN,NaN
1,60.00,40.00,872.16,789.46,NaN,NaN,NaN,NaN,NaN
2,70.00,30.00,1017.52,592.10,NaN,NaN,NaN,NaN,NaN
3,70.25,29.75,1021.15,587.17,NaN,NaN,NaN,NaN,NaN
4,80.00,20.00,1162.88,394.73,NaN,NaN,NaN,NaN,NaN


Now that's sorted, we can calculate the following data as per the csv file:

#### 1. **Density** ($\rho$)

We'll be using the following formula:

$$
\rho_p = \frac{1}{\frac{f_0}{\rho_0}+\frac{f_1}{\rho_1}}
$$

where $f_n$ is the **mass fraction** of the propellant component and $\rho_n$ is the respective density.

#### 2. **Chamber $c_p/c_v$** ($k$ or $\gamma$)

This is the **specific heat ratio** and it is used to determine the chamber pressure. It's given by:

$$
k = \frac{1}{1-\frac{R}{\frac{X}{1-X}C_s+C_p}}
$$

where $R$ is the **universal gas constant**, $\frac{X}{1-X}$ is the **mole fraction of condensed phase products**, $C_s$ is the **specific heat of the mixture of condensed phase products** and $C_p$ is the **specific heat of the mixture of gaseous products**.

It should be noted that this is for a **2-phase flow** where we have both condensed(*solid*) and gaseous phase products from this combustion.(Ikiara, 2025)

#### 3. **Chamber velocity** ($C^*$)

This is essential in determining the speed at which the propellant will burn at. It's given by:

$$
C^*=\sqrt{\frac{R\:T_o}{k\left(\frac{2}{k+1}\right)^{\frac{k+1}{k-1}}}}
$$

where $T_o$ is the **chamber temperature**.

#### 4. **Specific impulse** ($I_{sp}$)

This is the crucial factor that determines the thrust of our SRM. It is given by:

$$
I_{sp}=\frac{1}{g}\sqrt{2T_o\left(\frac{R}{M}\right)(\frac{k}{k-1})\left[1-\frac{P_e}{P_o}\right]^{\frac{k-1}{k}}}
$$

where $g$ is the **accelearation due to gravity**, $M$ is the **molecular mass of the KNSB grain**, $P_e$ and $P_o$ are the **exit** and **chamber pressures**.

#### 5. Equilibrium Analysis

This represents the state at which the propellant mixture undergoes complete combustion, achieving thermodynamic equilibrium where reactants are fully converted to products and the system reaches a stable energetic state.

We'll load a $mol$ of the propellant mix and equilibriate at adiabatic flame temperature:

In [ ]:
e = ppp.Equilibrium()
e.add_propellants_by_mass([(kno3, 944.84), (sorb, 690.78)])
ppr.pprint(e)

Status:
	Equillibrium Computed: False
	Properties Computed: False
	Performance Computed: False
Composition:
	POTASSIUM NITRATE - 9.345 mol
	SORBITOL - 3.792 mol
State:
	Pressure: 0.000 atm 
	Temperature: 0.0 K 
	Enthalpy: 0.000 kJ/kg 
	Int. Energy: 0.000 kJ/kg 
	Gibbs Free Energy: 0.000 kJ/kg 
	Entropy: 0.000 kJ/kg-K 
	Molar Mass: 0.000 g/mol 
	dV_P: 0.000
	dV_T: 0.000
	Cp: 0.000 kJ/kg-K
	Cv: 0.000 kJ/kg-K
	gamma: 0.000
	Sound Speed: 0.0 m/s
	
	


We'll perform a calculation at the point of complete combustion as follows:

In [1]:
e.reset()
R_universal = 8.314
T = np.linspace(1500, 3000, 100)
components = {
    "H2O"   : [],
    "CO2"   : [],
    "CO"    : [],
    "K2CO3" : [],
    "KOH"   : [],
    "N2"    : [],
    "H2"    : []
}
for i in range(len(T)):
    e.set_state(P=1., T=T[i], type="TP")
    # U[i] = e.properties.U
    components["H2O"].append(e.composition["H2O"])
    components["CO2"].append(e.composition["CO2"])
    components["CO"].append(e.composition["CO"])
    components["H2"].append(e.composition["H2"])
    components["K2CO3"].append(e.composition["K2CO3"])
    components["KOH"].append(e.composition["KOH"])
    components["N2"].append(e.composition["N2"])
    e.reset()

for lbl,c in components.items():
    plt.plot(T, c, label=lbl)
    plt.xlabel('Temperature (K)')
    plt.ylabel('Mole Fraction')
    plt.legend(fontsize=8, loc='upper right', title='Species', title_fontsize=10)

plt.suptitle('KNSB Composition during combustion', fontweight='bold')
output_dir = '../share/images'
os.makedirs(output_dir, exist_ok=True)
plt.savefig(os.path.join(output_dir, 'KNSB_Compostion_During_Combustion.png'), bbox_inches='tight')
plt.show()

NameError: name 'e' is not defined

From this data we can show the relationship between $I_{sp}$ and mass of $KNO_3$ used:

In [ ]:
p = ppp.ShiftingPerformance()
p.add_propellants_by_mass([(sorb, m_fu), (kno3, m_ox)])
p.set_state(P=P_CHAMBER_ATM, Pe=P_EXIT_ATM)

isps.append(round(p.performance.Isp, 4))
chamber_temps.append(round(p.properties[0].T, 4))
throat_temps.append(round(p.properties[1].T, 4))
exit_temps.append(round(p.properties[2].T, 4))
c_stars.append(round(p.performance.cstar, 4))
cps.append(round(p.properties[0].Cp, 4))
cvs.append(round(p.properties[0].Cv, 4))
isexs.append(round(p.properties[0].Isex, 4))
gammas.append(round(p.properties[0].Cp / p.properties[0].Cv, 4))
for _, row in df.iterrows():
    pct_ox = row['KNO3 (%)']
    pct_fu = row['Sorbitol (%)']
    m_ox    = row['Mass KNO3 (g)']
    m_fu    = row['Mass Sorbitol (g)']
    
    try:

        p.reset()
    except Exception as e:
        ppr.pprint(f"[FAIL] KNO3={pct_ox:.2f}% Sorbitol={pct_fu:.2f}% — solver error: {e}")
        continue
    
    of_ratio = m_ox / m_fu if m_fu != 0 else 0
    of_ratios.append(round(of_ratio, 4))

    f_ox  = pct_ox   / 100.0
    f_fu  = pct_fu / 100.0
    rho_p = 1.0 / (f_ox / RHO_KNO3 + f_fu / RHO_SORBITOL)
    densities.append(round(rho_p, 4))

pprint(f"[INFO] Simulation completed for {len(isps)} cases.")
pprint(f"[INFO] Chamber composition: {p.composition['chamber']}")
pprint(f"[INFO] Throat composition: {p.composition['throat']}")
pprint(f"[INFO] Exit composition: {p.composition['exit']}")

NameError: name 'df' is not defined

We'll then write the results back to the DataFrame and display it for confirmation:

In [ ]:
n = min(len(df), len(c_stars), len(densities), len(cps), len(cvs), len(chamber_temps), len(throat_temps), len(exit_temps), len(isps), len(of_ratios))
df = df.iloc[:n].copy()

df['C*']            = c_stars[:n]
df['Density']       = densities[:n]
df['Chamber Cp']    = cps[:n]
df['Chamber Cv']    = cvs[:n]
df['Chamber Temp']  = chamber_temps[:n]
df['Throat Temp']   = throat_temps[:n]
df['Exit Temp']     = exit_temps[:n]
df['Isp*']          = isps[:n]
df['Chamber Cp/Cv'] = gammas[:n]
df['O/F']           = of_ratios[:n]

df.to_csv("../share/csv/Thermochemical_Justification.csv", index=False)
df.head()

,KNO3 (%),Sorbitol (%),Mass KNO3 (g),Mass Sorbitol (g),C*,Density,Chamber Cp/Cv,Chamber Temp,Isp*,Molecular Weight,Chamber Cp,Chamber Cv,Throat Temp,Exit Temp,Isex,O/F,Performance Ratio
0,55.00,45.00,799.48,888.15,836.598,1.7762,1.2259,1117.0942,1340.5511,NaN,5.8641,4.7834,1028.8268,570.5324,1.1303,0.9002,1186.013536
1,55.25,44.75,803.11,883.22,836.598,1.7777,1.2259,1117.0942,1340.5511,NaN,5.8641,4.7834,1028.8268,570.5324,1.1303,0.9093,1186.013536
2,55.50,44.50,806.75,878.28,836.598,1.7793,1.2259,1117.0942,1340.5511,NaN,5.8641,4.7834,1028.8268,570.5324,1.1303,0.9186,1186.013536
3,55.75,44.25,810.38,873.35,836.598,1.7809,1.2259,1117.0942,1340.5511,NaN,5.8641,4.7834,1028.8268,570.5324,1.1303,0.9279,1186.013536
4,56.00,44.00,814.02,868.40,836.598,1.7824,1.2259,1117.0942,1340.5511,NaN,5.8641,4.7834,1028.8268,570.5324,1.1303,0.9374,1186.013536


We can also outline the relationship between the flame temperatures and mass of $KNO_3$ used:

In [ ]:
import os

opt_target = 1.857
opt_idx = df['O/F'].sub(opt_target).abs().idxmin()
opt_of = df.loc[opt_idx, 'O/F']

fig = plt.figure(figsize=(15, 4))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

panels = [
    (gs[0], isps,          'Specific Impulse (s)',          'Isp'),
    (gs[1], chamber_temps, 'Chamber Temperature (K)',        'T_c'),
    (gs[2], c_stars,       'Characteristic Velocity (m/s)', 'C*'),
]

for spec, y, ylabel, label in panels:
    ax = fig.add_subplot(spec)
    n = min(len(of_ratios), len(y))
    ax.plot(of_ratios[:n], y[:n], 'o-', lw=1.5, ms=4, color='steelblue', label=label)
    ax.axvline(opt_of, color='tomato', ls='--', lw=1.2,
               label=f'Optimum O/F = {opt_of:.3f}')
    ax.set_xlabel('O/F Ratio (KNO₃ / Sorbitol)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

plt.suptitle('KNSB Performance vs. O/F Ratio', fontweight='bold')
plt.tight_layout()

# Create directories if they don't exist
output_dir = '../share/images'
os.makedirs(output_dir, exist_ok=True)

# Save figure
plt.savefig(os.path.join(output_dir, 'knsb_performance_of_ratio.png'), dpi=150, bbox_inches='tight')
plt.show()

IndexError: list index out of range

---
## 4. What we can learn from this data

We'll find that with more mass of $KNO_3$ used, do these critical factors i.e. $I_{sp}$, chamber temperature and $C^*$ increase. In reality, KNSB grains made with such concentrations of $KNO_3$ are harder to process and for the motor casing to handle.

We'll also find that with less mass of $KNO_3$ used, the motor is cooler but less energetic. We can see that at $65\%:35\%$ i.e. when mass of $KNO_3$ used is $944.84g$, we have the "[sweet spot](https://tenor.com/blpXi.gif)".

It's also worth mentioning that when too much oxidizer is used, the KNSB grain becomes brittle and sensitive to moisture while when too much fuel is used, incomplete combustion and lower $I_{sp}$ is noted. ([Nakka, 2025](https://www.nakka-rocketry.net/sorb.html))

Since it's a 2-phase flow analysis:
### 1. The gaseous phase
The dominant species from the combustion tell us the following in the table below:
| Species | Mole fraction | What it means |
| :--: | :--: | :--: |
| $H_2O$ | 0.304 | Primary hydrogen combustion product |
| $CO$ | 0.185 | Incomplete carbon oxidation |
| $H_2$ | 0.176 | Unburned hydrogen |
| $CO_2$ | 0.122 | Complete carbon oxidation |
| $N_2$ | 0.106 | Nitrogen release from $KNO_3$ |

In [ ]:
# Quick sanity check
# After the loop, run this for the optimal ratio point
p_opt = ppp.FrozenPerformance()
p_opt.add_propellants_by_mass([
    (kno3,     df.loc[opt_idx, 'Mass KNO3 (g)']),
    (sorb, df.loc[opt_idx, 'Mass Sorbitol (g)']),
])
p_opt.set_state(P=68., Pe=1.0)

print("=== Optimal KNSB (O/F = {:.3f}) ===".format(opt_of))
print("\nGaseous chamber products:")
import pprint
pprint.pprint(p_opt.composition['chamber'][:8])
print("\nCondensed chamber products (K₂CO₃ should appear here):")
pprint.pprint(p_opt.composition_condensed['chamber'])

---
## 5. What's in-store for the future of this simulation
We plan on performing a CFD Simulation of 1-D isentropic nozzle flow i.e.:

$$
M_e = f(P_c, P_e, \gamma), v_e = \sqrt{\frac{2\gamma}{\gamma-1}\frac{RT_e}{M_w}\left[1-\left(\frac{P_e}{P_c}\right)^{\frac{\gamma-1}{\gamma}}\right]}
$$

using OpenFOAM and ANSYS programs hence the 2 folders in the `/share` directory. This is pretty taxing so far and will require some time.


---
## 6. In conclusion...

We justified the optimal KNSB ratio i.e. $65\%:35\%$, as it provides a high specific impulse with efficient combustion, *indicated by $I_{sp}$ and $C^*$*, while maintaining safe thermal characteristics and physical integrity. It is a well-characterized formulation that makes it a reliable and practical choice for our SRM.

----
## References

[1] R. Nakka, "KNSB Propellant," Richard Nakka's Experimental Rocketry Web Site, 2025. [Online]. Available: https://www.nakka-rocketry.net/sorb.html

[2] R. Nakka, "Solid Rocket Motor Theory - Two-phase flow," Richard Nakka's Experimental Rocketry Web Site, 2025. [Online]. Available: https://www.nakka-rocketry.net/th_2phf.htm

[3] R. Nakka, "Solid Rocket Motor Theory -- Impulse and C-star," Richard Nakka's Experimental Rocketry Web Site, 2025. [Online]. Available: https://www.nakka-rocketry.net/th_imp.html

[4] R. Nakka, "Two-phase flow theory," Richard Nakka's Experimental Rocketry Web Site, Oct. 28, 2011. [Online]. Available: nakka-rocketry.net. [Accessed: May 31, 2026].

[5] R. Nakka, "Potassium nitrate/sorbitol (KNSB) propellant performance characteristics," Richard Nakka's Experimental Rocketry Web Site, Mar. 22, 2011. [Online]. Available: nakka-rocketry.net. [Accessed: May 31, 2026].

[5] J. Bonnie, J. Zehe, and S. Gordon, "NASA Glenn Coefficients for Calculating Thermodynamic Properties of Individual Species," NASA/TP—2002-211556, Glenn Research Center, Cleveland, 2002.

[6] T. McReary, Experimental Composite Propellant: An Introduction To Properties And Preparation OF Composite Propellants: Design, Construction, Testing, and Characteristics Of Small Rocket Motors, 1st ed. 2020.

[7] D. Mazza and E. Canuto, Fundamental Chemistry with MATLAB. Oxford Publishing, 2022.

[8] B. G. Ikiara, "Preparation of KNSB Composite Sugar Propellant: An Investigation and Commentary," Nakuja Project, 2025. [Online]. Available: https://. [Accessed: Jun. 1, 2026].
